### This script identifies known colocalisations between sle gwas and eQTL/sc-eQTLs in the OTAR database
## all data is downloaded from https://platform.opentargets.org/downloads/study/schema
## use conda envionrment spark_env(Python 3.14.2) in local (java and spark versions are compatible) - may work with module load HGI/softpack/users/hj10/SLEmap_HJ/12 (have not tried)
## base directory: /Users/hj10/Desktop/PhDproject/1_SLEmap/Analysis/coloc/OTAR_COLOC

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
from pyspark.sql.functions import array, lit, col


In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("LocalOntologyIndexing") \
    .config("spark.executor.memory", "8g") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/29 09:36:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
! wget --recursive --no-parent --no-host-directories --cut-dirs 6 ftp://ftp.ebi.ac.uk/pub/databases/opentargets/platform/26.03/output/credible_set .
! wget --recursive --no-parent --no-host-directories --cut-dirs 6 ftp://ftp.ebi.ac.uk/pub/databases/opentargets/platform/26.03/output/study/ .
! rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/colocalisation .

--2026-03-29 09:36:57--  ftp://ftp.ebi.ac.uk/pub/databases/opentargets/platform/26.03/output/credible_set
           => '.listing'
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /pub/databases/opentargets/platform/26.03/output ... done.
==> PASV ... done.    ==> LIST ... done.

.listing                [ <=>                ]   4.21K  --.-KB/s    in 0.005s  

2026-03-29 09:36:57 (861 KB/s) - '.listing' saved [4311]

Removed '.listing'.
--2026-03-29 09:36:57--  ftp://ftp.ebi.ac.uk/pub/databases/opentargets/platform/26.03/output/credible_set/credible_set
           => 'credible_set/.listing'
==> CWD (1) /pub/databases/opentargets/platform/26.03/output/credible_set ... done.
==> PASV ... done.    ==> LIST ... done.

credible_set/.listi     [ <=>                ]  24.60K  --.-KB/s    in 0.05s  

In [4]:
! rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/colocalisation .


Transfer starting: 202 files
deleting colocalisation/part-00024-a74dd634-ab6d-4a54-94c2-70d5a01ab7dd-c000.snappy.parquet
deleting colocalisation/part-00023-a74dd634-ab6d-4a54-94c2-70d5a01ab7dd-c000.snappy.parquet
deleting colocalisation/part-00022-a74dd634-ab6d-4a54-94c2-70d5a01ab7dd-c000.snappy.parquet
deleting colocalisation/part-00021-a74dd634-ab6d-4a54-94c2-70d5a01ab7dd-c000.snappy.parquet
deleting colocalisation/part-00020-a74dd634-ab6d-4a54-94c2-70d5a01ab7dd-c000.snappy.parquet
deleting colocalisation/part-00019-a74dd634-ab6d-4a54-94c2-70d5a01ab7dd-c000.snappy.parquet
deleting colocalisation/part-00018-a74dd634-ab6d-4a54-94c2-70d5a01ab7dd-c000.snappy.parquet
deleting colocalisation/part-00017-a74dd634-ab6d-4a54-94c2-70d5a01ab7dd-c000.snappy.parquet
deleting colocalisation/part-00016-a74dd634-ab6d-4a54-94c2-70d5a01ab7dd-c000.snappy.parquet
deleting colocalisation/part-00015-a74dd634-ab6d-4a54-94c2-70d5a01ab7dd-c000.snappy.parquet
deleting colocalisation/part-00014-a74dd634-ab6d-4a

In [5]:
## Read in study info
study = spark.read.parquet("file:///Users/hj10/Desktop/PhDproject/1_SLEmap/Analysis/coloc/otar_coloc/study")
study.show(5, truncate=False)

## from study info, filter to SLE gwas studies
study_MONDO_SLE_only = study.filter(
    col("traitFromSourceMappedIds") == array(lit("MONDO_0007915"))
)
study_MONDO_SLE_only = study_MONDO_SLE_only.filter(
    col("studyType") == "gwas"
)
study_MONDO_SLE_only.show(5, truncate=False)
study_MONDO_SLE_only.count()

26/03/29 10:08:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------------------------+------+-----------+---------+----------------------------------------------------+------------------------+---------------------+--------+----------------------------------------------------------------------------------------------------+----------------------+---------------+------------------+----------------------------------+-------------------------------------------+------+---------+--------+---------+---------------------+-------------------+------------------+----------------------------------------------------------+-------------+------------------------------------------------------------------------------------+-----------+---------+---------------+---------------+--------------------+-----------+
|studyId                               |geneId|projectId  |studyType|traitFromSource                                     |traitFromSourceMappedIds|biosampleFromSourceId|pubmedId|publicationTitle                                               

128

In [6]:
## Read in study-locus info and get credible sets for sle gwas studies

credible_set = spark.read.parquet("file:///Users/hj10/Desktop/PhDproject/1_SLEmap/Analysis/coloc/otar_coloc/credible_set")
credible_set.show(5, truncate=False)

studylocus_SLE_only = credible_set.join(
    study_MONDO_SLE_only.select(["studyID","traitFromSource"]).distinct(),
    on="studyID",
    how="inner"
)
studylocus_SLE_only.show(5, truncate=False)
studylocus_SLE_only.count()

+--------------------------------+--------------------------------------------------------------------+----------------+----------+--------+-----------------------+---------+------+--------------+--------------+-------------------------------+-------------+-------------------+---------------+-----------------+----------------+------------------+------------+-----------+----------+--------+----------+-----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

+------------+--------------------------------+---------------+----------+--------+------+---------+------+--------------+--------------+-------------------------------+--------------------+-------------------+----------------------------------------------------------------------------------------------------------------------------+-----------------+----------------+------------------+------------+-----------+----------+--------+----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

1659

In [7]:
# Read in colocalisation results and filter to just eqtls and sceqtls and filter to sig colocs
colocs = spark.read.parquet("file:///Users/hj10/Desktop/PhDproject/1_SLEmap/Analysis/coloc/otar_coloc/colocalisation")
colocs = colocs.filter(col("rightStudyType").isin("eqtl","sceqtl"))
colocs = colocs.filter(col("h4") > 0.8)


In [8]:
## filter to colocs between SLE gwas and eQTL/sc-eQTLs
colocs_SLE_only = colocs.join(
    studylocus_SLE_only.select(["studyID","studyLocusId",'variantId',"traitFromSource",'finemappingMethod']),
    colocs.leftStudyLocusId == studylocus_SLE_only.studyLocusId,
    how="inner"
)
colocs_SLE_only  = colocs_SLE_only.withColumnRenamed("studyID", "GWAS_studyID")
colocs_SLE_only = colocs_SLE_only.drop("studyLocusId")

colocs_SLE_only.show(5, truncate=False)
colocs_SLE_only.count()

+--------------------------------+--------------------------------+--------------+----------+--------------------+--------------------------+-------------------+------------------+--------------------+--------------------+------------+---------------+-----------------------------------+-----------------+
|leftStudyLocusId                |rightStudyLocusId               |rightStudyType|chromosome|colocalisationMethod|numberColocalisingVariants|h3                 |h4                |clpp                |betaRatioSignAverage|GWAS_studyID|variantId      |traitFromSource                    |finemappingMethod|
+--------------------------------+--------------------------------+--------------+----------+--------------------+--------------------------+-------------------+------------------+--------------------+--------------------+------------+---------------+-----------------------------------+-----------------+
|3534640ff4f0847c62f6d9193d7db92e|a0a1d626f9a6067e44ddc57f435f3d79|eqtl          |

21211

In [9]:
### get eQTL study ID for sle colocs 
credible_subset = credible_set.select("studyLocusId", "studyID")
colocs_SLE_only = colocs_SLE_only.join(
    credible_subset,
    colocs_SLE_only.rightStudyLocusId == credible_subset.studyLocusId,
    how="left"  # or "inner" if you only want matches
)
colocs_SLE_only  = colocs_SLE_only.withColumnRenamed("studyID", "eQTL_studyID")
colocs_SLE_only = colocs_SLE_only.drop("studyLocusId")
colocs_SLE_only.show(5)

+--------------------+--------------------+--------------+----------+--------------------+--------------------------+-------------------+------------------+--------------------+--------------------+------------+--------------+--------------------+-----------------+--------------------+
|    leftStudyLocusId|   rightStudyLocusId|rightStudyType|chromosome|colocalisationMethod|numberColocalisingVariants|                 h3|                h4|                clpp|betaRatioSignAverage|GWAS_studyID|     variantId|     traitFromSource|finemappingMethod|        eQTL_studyID|
+--------------------+--------------------+--------------+----------+--------------------+--------------------------+-------------------+------------------+--------------------+--------------------+------------+--------------+--------------------+-----------------+--------------------+
|c09779da16f96bf21...|2a18d7f47b72972b3...|          eqtl|         8|   COLOC_PIP_ECAVIAR|                         1|0.02354608783587638|0.

In [10]:
### get eQTL study info for sle colocs 

study_subset = study.select("studyId","geneId","projectId","pubmedId")
colocs_SLE_only = colocs_SLE_only.join(
    study_subset,
    colocs_SLE_only.eQTL_studyID == study_subset.studyId,
    how="left"  # or "inner" if you only want matches
)
colocs_SLE_only = colocs_SLE_only.drop("studyId")
colocs_SLE_only.show(5)
colocs_SLE_only.count()

+--------------------+--------------------+--------------+----------+--------------------+--------------------------+-------------------+------------------+--------------------+--------------------+------------+---------------+--------------------+-----------------+--------------------+---------------+---------+--------+
|    leftStudyLocusId|   rightStudyLocusId|rightStudyType|chromosome|colocalisationMethod|numberColocalisingVariants|                 h3|                h4|                clpp|betaRatioSignAverage|GWAS_studyID|      variantId|     traitFromSource|finemappingMethod|        eQTL_studyID|         geneId|projectId|pubmedId|
+--------------------+--------------------+--------------+----------+--------------------+--------------------------+-------------------+------------------+--------------------+--------------------+------------+---------------+--------------------+-----------------+--------------------+---------------+---------+--------+
|bfa503cbb55d1c782...|122c1f024

84844

In [11]:
# Write to TSV
colocs_SLE_only.toPandas().to_csv("/Users/hj10/Desktop/PhDproject/1_SLEmap/Analysis/coloc/otar_coloc/coloc_alreadyknown_otar_26.03.csv", index=False)

In [12]:
colocs_SLE_variant = colocs_SLE_only.dropDuplicates(["variantId"])
colocs_SLE_variant.show(10)
colocs_SLE_variant.count()

+--------------------+--------------------+--------------+----------+--------------------+--------------------------+--------------------+------------------+--------------------+--------------------+------------+----------------+--------------------+-----------------+--------------------+---------------+----------+--------+
|    leftStudyLocusId|   rightStudyLocusId|rightStudyType|chromosome|colocalisationMethod|numberColocalisingVariants|                  h3|                h4|                clpp|betaRatioSignAverage|GWAS_studyID|       variantId|     traitFromSource|finemappingMethod|        eQTL_studyID|         geneId| projectId|pubmedId|
+--------------------+--------------------+--------------+----------+--------------------+--------------------------+--------------------+------------------+--------------------+--------------------+------------+----------------+--------------------+-----------------+--------------------+---------------+----------+--------+
|f6417b307c18972f7...|

323

In [ ]:
colocs_SLE_variant.toPandas().to_csv("/Users/hj10/Desktop/PhDproject/1_SLEmap/Analysis/coloc/otar_coloc/coloc_alreadyknown_otar_uniquevariants_26.03.csv", index=False)

26/03/29 13:39:42 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 120225 ms exceeds timeout 120000 ms
26/03/29 13:39:42 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/29 13:39:47 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$